<a href="https://colab.research.google.com/github/JeysonCarmona/PPMI_INVESTIGATION/blob/main/notebook3_images_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 — Image Analysis (PPMI)

**Objective:** Read only the **internal directory tree** of `Imagenes_PPMI.rar` (183 GB compressed / 560 GB uncompressed / +3M files) **without extracting content**, automatically detect patients, modalities, views, studies, and dates from the path names, and generate `imagenes_index.csv`.

DICOM files are not opened in this notebook. Only the internal names of the RAR (header metadata) are read, which is much lighter than decompressing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# El backend 'unrar' es necesario para que la librería rarfile pueda leer
# el índice interno del .rar sin descomprimir el contenido.
!apt-get install -y unrar -qq
!pip install rarfile -q


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Investigación_Parkinson"
RAR_IMAGENES = BASE_DIR + "/Imagenes/Imagenes_PPMI.rar"
RESULTADOS_DIR = BASE_DIR + "/Jeyson_Carmona_Michael_Lamprea/segunda entrega/Resultados"

import os
os.makedirs(RESULTADOS_DIR, exist_ok=True)

assert os.path.exists(RAR_IMAGENES), f"No se encontró el archivo RAR: {RAR_IMAGENES}"
print("Archivo RAR OK:", RAR_IMAGENES)
print("Tamaño en disco: {:.2f} GB".format(os.path.getsize(RAR_IMAGENES) / 1024**3))


Archivo RAR OK: /content/drive/MyDrive/Investigación_Parkinson/Imagenes/Imagenes_PPMI.rar
Tamaño en disco: 183.84 GB


In [ ]:
import rarfile
import re
import pandas as pd
from collections import Counter

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

rarfile.UNRAR_TOOL = "unrar"


## 1. Reading the internal RAR index (without extracting)

The `unrar lb -r` command (list bare, recursive) is used via `subprocess`, which only reads the internal RAR headers (path names) without decompressing the content of any file. This avoids the previous problem where `rarfile.RarFile(...).infolist()` could end up generating writes to Colab's temporary disk when processing such a large multi-volume RAR over a Google Drive mount. With +3 million entries, this cell can take several minutes: it is still only metadata reading, not content.

In [ ]:
# DIAGNÓSTICO: la causa más probable de que el disco temporal de Colab se llene
# es que rarfile.RarFile(...) + rf.infolist() invoca internamente al binario
# 'unrar' en un modo de listado "verbose" (comando 'v'/'vt') que, sobre archivos
# multivolumen/solid muy grandes leídos desde un montaje FUSE de Google Drive,
# puede recurrir a caché/temporales en /tmp o /content al recorrer cabeceras.
#
# La forma garantizada de listar SIN descomprimir ni escribir nada en disco es
# usar directamente el comando 'unrar lb' (list bare = solo nombres de archivo,
# sin extraer contenido), leyendo la salida en streaming línea por línea.

import subprocess

def listar_rutas_rar(ruta_rar, unrar_path="unrar"):
    """Lee únicamente los nombres internos del RAR usando 'unrar lb -r'.
    Este modo SOLO lee cabeceras y devuelve nombres de ruta: no extrae,
    no descomprime contenido y no escribe archivos temporales en disco.
    Se usa streaming línea a línea para no acumular toda la salida en memoria de golpe.
    """
    comando = [unrar_path, "lb", "-r", ruta_rar]  # lb = list bare, -r = recursivo
    proceso = subprocess.Popen(
        comando,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )

    rutas_archivos = []
    for linea in proceso.stdout:
        linea = linea.rstrip("\n").replace("\\", "/")
        if not linea:
            continue
        # 'lb' puede incluir entradas de carpeta (terminan en '/' o no tienen
        # extensión); las descartamos igual que antes hacía info.isdir().
        if linea.endswith("/"):
            continue
        rutas_archivos.append(linea)

    proceso.stdout.close()
    codigo_retorno = proceso.wait()
    stderr_out = proceso.stderr.read()
    proceso.stderr.close()

    if codigo_retorno != 0:
        raise RuntimeError(f"unrar devolvió código {codigo_retorno}. Detalle: {stderr_out}")

    return rutas_archivos

print("Función de listado (solo metadatos, sin extraer) definida.")


Función de listado (solo metadatos, sin extraer) definida.


In [ ]:
# Ejecuta el listado. Con +3M entradas puede tardar varios minutos: es
# lectura de cabeceras vía 'unrar lb', nunca descompresión de contenido.
rutas = listar_rutas_rar(RAR_IMAGENES)
print("Rutas de archivo dentro del RAR (excluyendo carpetas):", len(rutas))


Rutas de archivo dentro del RAR (excluyendo carpetas): 3277408


In [ ]:
print("Ejemplos de rutas:")
for r in rutas[:5]:
    print(" -", r)


Ejemplos de rutas:
 - Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_AXIAL_2D/PPMI/100956/rsfMRI_LR/2021-06-14_11_49_08.0/I11083239/PPMI_100956_MR_rsfMRI_LR__br_raw_20241224091221873_1.dcm
 - Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_AXIAL_2D/PPMI/100956/rsfMRI_LR/2021-06-14_11_49_08.0/I11083239/PPMI_100956_MR_rsfMRI_LR__br_raw_20241224091221919_10.dcm
 - Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_AXIAL_2D/PPMI/100956/rsfMRI_LR/2021-06-14_11_49_08.0/I11083239/PPMI_100956_MR_rsfMRI_LR__br_raw_20241224091222005_2.dcm
 - Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_AXIAL_2D/PPMI/100956/rsfMRI_LR/2021-06-14_11_49_08.0/I11083239/PPMI_100956_MR_rsfMRI_LR__br_raw_20241224091222080_3.dcm
 - Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_AXIAL_2D/PPMI/100956/rsfMRI_LR/2021-06-14_11_49_08.0/I11083239/PPMI_100956_MR_rsfMRI_LR__br_raw_20241224091222364_4.dcm


## 2. Automatic parsing of path structure

Instead of assuming a fixed folder depth (which can vary), patterns are detected by keywords within each path:

- **Diagnostic Group:** `Control` / `Parkinson_Disease` (or `PD`)
- **View:** `Axial` / `Sagital`
- **Sequence:** `T1` / `T2`
- **Dimension:** `2D` / `3D`
- **Patient (PATNO):** numeric folder located just after a folder named `PPMI`
- **Study/Date:** folders following the PATNO (study description and date in `YYYY-MM-DD` format)

In [ ]:
PATRON_GRUPO = re.compile(r"(Control|Parkinson_Disease|PD)", re.IGNORECASE)
PATRON_VISTA = re.compile(r"(Axial|Sagital)", re.IGNORECASE)
PATRON_SECUENCIA = re.compile(r"(T1|T2)", re.IGNORECASE)
PATRON_DIMENSION = re.compile(r"(2D|3D)", re.IGNORECASE)
PATRON_FECHA = re.compile(r"^\d{4}-\d{2}-\d{2}$")

def parsear_ruta(ruta):
    """Extrae metadatos de una ruta interna del RAR usando los segmentos de carpeta.
    No asume profundidad fija: busca patrones por palabras clave.
    """
    partes = ruta.split("/")

    grupo = None
    vista = None
    secuencia = None
    dimension = None
    patno = None
    estudio_desc = None
    fecha = None

    m = PATRON_GRUPO.search(ruta)
    if m:
        grupo = m.group(1)
    m = PATRON_VISTA.search(ruta)
    if m:
        vista = m.group(1)
    m = PATRON_SECUENCIA.search(ruta)
    if m:
        secuencia = m.group(1)
    m = PATRON_DIMENSION.search(ruta)
    if m:
        dimension = m.group(1)

    # Paciente: carpeta numérica justo después de una carpeta llamada "PPMI"
    for i, parte in enumerate(partes):
        if parte.upper() == "PPMI" and i + 1 < len(partes):
            posible_patno = partes[i + 1]
            if posible_patno.isdigit():
                patno = int(posible_patno)
                # La carpeta siguiente al PATNO suele ser la descripción del estudio
                if i + 2 < len(partes):
                    estudio_desc = partes[i + 2]
                # La carpeta siguiente a la descripción suele ser la fecha
                if i + 3 < len(partes) and PATRON_FECHA.match(partes[i + 3]):
                    fecha = partes[i + 3]
            break

    return {
        "ruta_completa": ruta,
        "grupo_diagnostico": grupo,
        "vista": vista,
        "secuencia": secuencia,
        "dimension": dimension,
        "PATNO": patno,
        "estudio": estudio_desc,
        "fecha": fecha,
        "nombre_archivo": partes[-1] if partes else None,
    }


In [ ]:
registros_imagenes = []
for i, ruta in enumerate(rutas):
    registros_imagenes.append(parsear_ruta(ruta))
    if (i + 1) % 500000 == 0:
        print(f"Procesadas {i + 1:,} rutas...")

df_rutas = pd.DataFrame(registros_imagenes)
print("\nTotal de rutas parseadas:", len(df_rutas))
df_rutas.head()


Procesadas 500,000 rutas...
Procesadas 1,000,000 rutas...
Procesadas 1,500,000 rutas...
Procesadas 2,000,000 rutas...
Procesadas 2,500,000 rutas...
Procesadas 3,000,000 rutas...

Total de rutas parseadas: 3277408


,ruta_completa,grupo_diagnostico,vista,secuencia,dimension,PATNO,estudio,fecha,nombre_archivo
0,Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_A...,Control,Axial,None,2D,100956.0,rsfMRI_LR,None,PPMI_100956_MR_rsfMRI_LR__br_raw_2024122409122...
1,Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_A...,Control,Axial,None,2D,100956.0,rsfMRI_LR,None,PPMI_100956_MR_rsfMRI_LR__br_raw_2024122409122...
2,Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_A...,Control,Axial,None,2D,100956.0,rsfMRI_LR,None,PPMI_100956_MR_rsfMRI_LR__br_raw_2024122409122...
3,Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_A...,Control,Axial,None,2D,100956.0,rsfMRI_LR,None,PPMI_100956_MR_rsfMRI_LR__br_raw_2024122409122...
4,Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_A...,Control,Axial,None,2D,100956.0,rsfMRI_LR,None,PPMI_100956_MR_rsfMRI_LR__br_raw_2024122409122...


### Which paths failed to identify a PATNO? (incomplete paths or with a different structure)

In [ ]:
rutas_incompletas = df_rutas[df_rutas["PATNO"].isna()]
print("Rutas sin PATNO detectado:", len(rutas_incompletas))
if len(rutas_incompletas) > 0:
    print("\nEjemplos:")
    display(rutas_incompletas["ruta_completa"].head(10))


Rutas sin PATNO detectado: 89

Ejemplos:


,ruta_completa
220662,Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_A...
220663,Imagenes_PPMI/Control/PD/Axial/2D/CONTROL_PD_A...
220686,Imagenes_PPMI/Control/PD/SAGITAL/2D/CONTROL_PD...
233105,Imagenes_PPMI/Control/PD/SAGITAL/3D/CONTROL_PD...
233106,Imagenes_PPMI/Control/PD/SAGITAL/3D/CONTROL_PD...
250701,Imagenes_PPMI/Control/T1/AXIAL/2D/CONTROL_T1_A...
250702,Imagenes_PPMI/Control/T1/AXIAL/2D/CONTROL_T1_A...
252239,Imagenes_PPMI/Control/T1/AXIAL/3D/CONTROL_T1_A...
252240,Imagenes_PPMI/Control/T1/AXIAL/3D/CONTROL_T1_A...
308168,Imagenes_PPMI/Control/T1/SAGITAL/3D/CONTROL_T1...


### How many patients have images? How many studies does each patient have?

In [ ]:
df_validas = df_rutas[df_rutas["PATNO"].notna()].copy()
df_validas["PATNO"] = df_validas["PATNO"].astype(int)

n_pacientes_img = df_validas["PATNO"].nunique()
print("Pacientes únicos con imágenes:", n_pacientes_img)

estudios_por_paciente = df_validas.groupby("PATNO")["estudio"].nunique().rename("n_estudios")
print("\nEstudios por paciente (resumen estadístico):")
print(estudios_por_paciente.describe())


Pacientes únicos con imágenes: 1471

Estudios por paciente (resumen estadístico):
count    1471.000000
mean        4.219579
std         2.362127
min         1.000000
25%         3.000000
50%         3.000000
75%         5.000000
max        24.000000
Name: n_estudios, dtype: float64


### How many modalities exist and how are they distributed?

In [ ]:
for col in ["grupo_diagnostico", "vista", "secuencia", "dimension"]:
    print(f"--- {col} ---")
    print(df_validas[col].value_counts(dropna=False))
    print()


--- grupo_diagnostico ---
grupo_diagnostico
Parkinson_Disease    2913602
Control               363717
Name: count, dtype: int64

--- vista ---
vista
AXIAL      2301375
SAGITAL     754033
Axial       221911
Name: count, dtype: int64

--- secuencia ---
secuencia
None    1994657
T1       863084
T2       416023
t1         2148
t2         1407
Name: count, dtype: int64

--- dimension ---
dimension
2D    2521720
3D     755599
Name: count, dtype: int64



### Are there duplicate studies (same patient + study + date combination)?

In [ ]:
claves_estudio = df_validas.dropna(subset=["estudio", "fecha"]).drop_duplicates(
    subset=["PATNO", "estudio", "fecha"]
)
conteo_estudios = df_validas.dropna(subset=["estudio", "fecha"]).groupby(
    ["PATNO", "estudio", "fecha"]
).size().rename("n_archivos_dcm")

print("Combinaciones únicas paciente+estudio+fecha:", len(claves_estudio))
print("\nArchivos DCM promedio por estudio: {:.1f}".format(conteo_estudios.mean()))


Combinaciones únicas paciente+estudio+fecha: 0

Archivos DCM promedio por estudio: nan


## 3. Construction of `imagenes_index.csv`

One row per study (not per individual DICOM file) to keep the index light.

In [ ]:
imagenes_index = df_validas.dropna(subset=["estudio"]).groupby(
    ["PATNO", "grupo_diagnostico", "vista", "secuencia", "dimension", "estudio", "fecha"],
    dropna=False
).size().reset_index(name="n_archivos")

print("imagenes_index.csv -> filas (estudios):", len(imagenes_index))
imagenes_index.head(10)


imagenes_index.csv -> filas (estudios): 6441


,PATNO,grupo_diagnostico,vista,secuencia,dimension,estudio,fecha,n_archivos
0,3000,Control,AXIAL,T2,2D,AX_T2_FLAIR,NaN,21
1,3000,Control,SAGITAL,T1,3D,sag_3D_FSPGR_BRAVO_straight,NaN,149
2,3001,Parkinson_Disease,AXIAL,T2,2D,AX_T2_AC-PC_line_Entire_Brain,NaN,81
3,3001,Parkinson_Disease,AXIAL,T2,2D,AX_T2_FLAIR_5_1,NaN,29
4,3001,Parkinson_Disease,SAGITAL,T1,3D,sag_3D_FSPGR_BRAVO_straight,NaN,155
5,3002,Parkinson_Disease,AXIAL,T2,2D,AX_T2_AC-PC_line_Entire_Brain,NaN,24
6,3002,Parkinson_Disease,AXIAL,T2,2D,AX_T2_FLAIR_5_1,NaN,28
7,3002,Parkinson_Disease,SAGITAL,T1,3D,sag_3D_FSPGR_BRAVO_straight,NaN,155
8,3003,Parkinson_Disease,AXIAL,T2,2D,AX_T2_AC-PC_line_Entire_Brain,NaN,21
9,3003,Parkinson_Disease,AXIAL,T2,2D,AX_T2_FLAIR_5_1,NaN,29


In [ ]:
ruta_salida = os.path.join(RESULTADOS_DIR, "imagenes_index.csv")
imagenes_index.to_csv(ruta_salida, index=False)
print("Guardado en:", ruta_salida)


Guardado en: /content/drive/MyDrive/Investigación_Parkinson/Resultados/imagenes_index.csv


## Executive Summary

In [ ]:
print("="*60)
print("RESUMEN — Notebook 3: Análisis de imágenes")
print("="*60)
print(f"Rutas de archivo totales dentro del RAR: {len(rutas):,}")
print(f"Rutas sin PATNO detectado: {len(rutas_incompletas):,}")
print(f"Pacientes únicos con imágenes: {n_pacientes_img}")
print(f"Estudios únicos (paciente+estudio+fecha): {len(claves_estudio):,}")
print(f"Archivo generado: {ruta_salida}")
print("="*60)


RESUMEN — Notebook 3: Análisis de imágenes
Rutas de archivo totales dentro del RAR: 3,277,408
Rutas sin PATNO detectado: 89
Pacientes únicos con imágenes: 1471
Estudios únicos (paciente+estudio+fecha): 0
Archivo generado: /content/drive/MyDrive/Investigación_Parkinson/Resultados/imagenes_index.csv
